# 07 · SQL Version of RFM + Power BI Data Export

**Part A:** load the clean data into a **SQL database** (SQLite) and rebuild RFM in pure SQL (`sql/rfm_analysis.sql`).
**Part B:** export a clean **star schema** of CSV files for the Power BI dashboard.

> 📘 **Why SQL too?** In a bank, data lives in databases/warehouses (HSBC mentions BigQuery and SQL in the JD). Analysts pull and shape data with SQL *before* Python or Power BI. Showing the same logic in both proves you can work at either layer.
>
> **SQLite** is a full SQL database stored in a single file, with no server to install. The SQL here (CTEs, window functions like `NTILE`, `LAG`, `SUM() OVER()`) is standard and transfers to PostgreSQL, SQL Server or BigQuery.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
PBI = ROOT / "powerbi" / "data"
PBI.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", "{:,.2f}".format)

sales = pd.read_parquet(PROCESSED / "sales_clean.parquet")

## Part A: SQL
### 1. Load clean sales into a SQLite database

In [2]:
db_path = ROOT / "data" / "retail.db"
con = sqlite3.connect(db_path)
cols = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price", "Revenue", "CustomerID", "Country"]
sql_sales = sales[cols].copy()
sql_sales["InvoiceDate"] = sql_sales["InvoiceDate"].dt.strftime("%Y-%m-%d %H:%M:%S")
sql_sales.to_sql("sales", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_customer ON sales(CustomerID)")
print(pd.read_sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT CustomerID) AS customers FROM sales", con))

      rows  customers
0  1003168       5852


### 2. Run the queries from `sql/rfm_analysis.sql`

In [3]:
queries = [q.strip() for q in (ROOT / "sql" / "rfm_analysis.sql").read_text(encoding="utf-8").split(";") if "SELECT" in q]
print(f"{len(queries)} queries found")

rfm_sql = pd.read_sql(queries[0], con)
rfm_sql.to_sql("rfm_sql", con, if_exists="replace", index=False)
rfm_sql.head()

4 queries found


,CustomerID,recency,frequency,monetary,r_score,f_score,m_score,segment
0,18102,0,145,"580,987.04",5,5,5,Champions
1,14646,1,145,"526,751.52",5,5,5,Champions
2,14156,9,144,"303,069.88",5,5,5,Champions
3,14911,1,373,"272,252.79",5,5,5,Champions
4,17450,8,51,"241,616.25",5,5,5,Champions


**Query 2: segment summary**

In [4]:
pd.read_sql(queries[1], con)

,segment,customers,avg_recency_days,avg_orders,revenue,pct_revenue
0,Champions,851,8.10,19.00,"8,862,533.00",52.90
1,Loyal Customers,1140,66.10,9.70,"4,691,698.00",28.00
2,At Risk,741,369.50,3.90,"1,035,003.00",6.20
3,Potential Loyalists,700,25.40,2.60,"572,386.00",3.40
4,Hibernating,1527,455.70,1.30,"536,518.00",3.20
5,Can't Lose Them,74,330.40,16.10,"534,145.00",3.20
6,Need Attention,276,109.20,3.10,"326,599.00",1.90
7,About to Sleep,379,106.50,1.30,"163,238.00",1.00
8,Promising,111,36.20,1.00,"20,401.00",0.10
9,New Customers,53,10.90,1.00,"10,982.00",0.10


**Query 3: monthly revenue & month-over-month growth (LAG window function)**

In [5]:
pd.read_sql(queries[2], con).tail(8)

,month,revenue,orders,mom_growth_pct
17,2011-05,"739,914.00",1668,47.70
18,2011-06,"736,726.00",1525,-0.40
19,2011-07,"688,145.00",1452,-6.60
20,2011-08,"724,197.00",1339,5.20
21,2011-09,"1,028,314.00",1818,42.00
22,2011-10,"1,099,106.00",2005,6.90
23,2011-11,"1,452,113.00",2751,32.10
24,2011-12,"446,018.00",815,-69.30


**Query 4: repeat-purchase rate by country**

In [6]:
pd.read_sql(queries[3], con).head(10)

,Country,customers,repeat_rate_pct
0,Switzerland,22,86.40
1,Portugal,23,78.30
2,Germany,107,77.60
3,Belgium,29,75.90
4,United Kingdom,5334,72.50
5,France,93,69.90
6,Spain,38,68.40
7,Netherlands,22,68.20


### 3. Reconcile SQL vs Python
**Reconciliation** (checking two systems give the same answer) is a daily task in Finance Operations. R, F, M raw values should match exactly. Segments may differ slightly because `NTILE` and `pd.qcut` break **ties** differently (e.g. many customers share Frequency = 1).

In [7]:
rfm_py = pd.read_parquet(PROCESSED / "rfm_clusters.parquet")
cmp = rfm_py.merge(rfm_sql, on="CustomerID")
print(f"Customers matched: {len(cmp):,} / {len(rfm_py):,}")
print(f"Recency identical:   {(cmp['Recency'] == cmp['recency']).mean():.1%}")
print(f"Frequency identical: {(cmp['Frequency'] == cmp['frequency']).mean():.1%}")
print(f"Monetary identical:  {np.isclose(cmp['Monetary'], cmp['monetary'], atol=0.01).mean():.1%}")
print(f"Segment agreement:   {(cmp['Segment'] == cmp['segment']).mean():.1%}  (differences = tie-breaking at quintile edges)")
con.close()

Customers matched: 5,852 / 5,852
Recency identical:   100.0%
Frequency identical: 100.0%
Monetary identical:  100.0%
Segment agreement:   90.6%  (differences = tie-breaking at quintile edges)


🔎 **A real reconciliation catch:** the first SQL version computed recency as `DATE(snapshot) − DATE(last purchase)` (calendar days), while Python counted *full 24-hour days elapsed*. Result: **0% of recency values matched**, every customer off by one day. Aligning the definition (subtract the full timestamp, truncate the fraction) fixed it. Lesson: when two systems disagree, the cause is usually a **definition** difference, not an arithmetic bug. Document definitions explicitly.

## Part B: Power BI export (star schema)
> 📘 **Star schema:** one central **fact table** (transactions, many rows) linked to **dimension tables** (customers, products, dates: descriptive attributes). It's the recommended model shape for Power BI: faster, simpler DAX, fewer errors.
>
> ```
>            DimDate (built in DAX)
>                 │
> DimProduct ── FactSales ── DimCustomer ── SegmentActions
> ```
> Plus standalone tables for cohort retention and the forecast.

In [8]:
# Power BI compares text case-INsensitively and ignores trailing spaces, but pandas doesn't: codes like
# "15056bl" vs "15056BL" (170 of them) and "47503J " vs "47503J" are the same product and would be duplicate
# keys in DimProduct → trim and upper-case them for the BI layer.
sales = sales.assign(StockCode=sales["StockCode"].str.strip().str.upper())

fact = sales[["Invoice", "InvoiceDate", "Date", "StockCode", "CustomerID", "Country", "Quantity", "Price", "Revenue"]].copy()
fact["CustomerID"] = fact["CustomerID"].astype("Int64")
fact.to_csv(PBI / "FactSales.csv", index=False)

product = (sales.groupby("StockCode")
           .agg(Description=("Description", lambda d: d.mode().iat[0] if d.notna().any() else "Unknown"),
                AvgPrice=("Price", "median"))
           .reset_index())
product.to_csv(PBI / "DimProduct.csv", index=False)

scores = pd.read_parquet(PROCESSED / "customer_scores.parquet")
cust = rfm_py.merge(scores[["CustomerID", "ChurnProbability", "RiskBand", "QuarterlyRevenue", "RevenueAtRisk"]],
                    on="CustomerID", how="left")
cust["RiskBand"] = cust["RiskBand"].astype("string").fillna("Inactive (>1 yr)")
cust = cust.drop(columns=["Cluster"])
cust.to_csv(PBI / "DimCustomer.csv", index=False)

seg = pd.read_csv(PROCESSED / "segment_actions.csv").drop(columns=["Customers", "% Revenue"])
seg["SortOrder"] = range(1, len(seg) + 1)  # best → worst, used for "Sort by column" in Power BI
seg.to_csv(PBI / "SegmentActions.csv", index=False)
pd.read_csv(PROCESSED / "cohort_retention.csv").to_csv(PBI / "CohortRetention.csv", index=False)
pd.read_csv(PROCESSED / "weekly_forecast.csv").to_csv(PBI / "WeeklyForecast.csv", index=False)

for f in sorted(PBI.glob("*.csv")):
    print(f"{f.name:22s} {len(pd.read_csv(f, usecols=[0])):>9,} rows  {f.stat().st_size/1e6:6.1f} MB")

# Key checks: dimension keys must be unique (case-insensitively, as Power BI sees them)
for name, df, keycol in [("DimProduct", product, "StockCode"), ("DimCustomer", cust, "CustomerID"), ("SegmentActions", seg, "Segment")]:
    dups = df[keycol].astype(str).str.strip().str.upper().duplicated().sum()
    print(f"{name}[{keycol}] duplicate keys: {dups}")
    assert dups == 0

CohortRetention.csv          325 rows     0.0 MB
DimCustomer.csv            5,852 rows     0.9 MB
DimProduct.csv             4,706 rows     0.2 MB


FactSales.csv          1,003,168 rows    79.2 MB
SegmentActions.csv            10 rows     0.0 MB
WeeklyForecast.csv           118 rows     0.0 MB
DimProduct[StockCode] duplicate keys: 0
DimCustomer[CustomerID] duplicate keys: 0
SegmentActions[Segment] duplicate keys: 0


## ✅ Key takeaways
1. The same RFM logic works in **SQL** with CTEs + window functions; raw R/F/M reconcile exactly with Python.
2. Small segment differences come from **tie-breaking**, a real-world reconciliation issue worth explaining.
3. Data is exported as a **star schema** for Power BI. See `powerbi/POWERBI_BUILD_GUIDE.md` for the dashboard steps.